In [1]:
import pandas as pd
import requests
import geopandas as gpd
from rasterstats import zonal_stats
import os, glob
import rasterio
import numpy as np
from rasterio.features import rasterize
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re

In [4]:
def get_key_from_environment(file_path: str, key: str) -> str | None:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")

In [ ]:
def get_data(scale):
  header = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json"
  }


  start_date = datetime(2020, 9, 1)   # Sept 2024
  end_date = datetime(2025, 8, 1)     # Aug 2025

  # Loop over months
  date = start_date
  while date <= end_date:
      date_str = date.strftime("%Y-%m")
      url = f"https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale{scale:03d}&date={date_str}"
      res = requests.get(url, headers=header)
          
      if res.status_code == 200:
          file = f"./data/spi{scale:03d}_{date_str}.tif"
          with open(file, "wb") as f:
              f.write(res.content)
          
      # Move to next month
      date += relativedelta(months=1)
    
get_data(3)

In [ ]:
# Diverging Stacked Area Chart.

In [12]:
thresholds = [
    (-np.inf, -2,   "#730000", "D4 Exceptional Drought"),
    (-2,      -1.6, "#FF0000", "D3 Extreme Drought"),
    (-1.6,    -1.3, "#FF9900", "D2 Severe Drought"),
    (-1.3,    -0.8, "#FFD37F", "D1 Moderate Drought"),
    (-0.8,    -0.5, "#FFFF00", "D0 Abnormally Dry"),
    (-0.5,     0.5, "#FFFFFF", "Near Normal"),
    ( 0.5,     0.8, "#99CCFF", "W0 Abnormally Wet"),
    ( 0.8,     1.3, "#0066CC", "W1 Moderately Wet"),
    ( 1.3,     1.6, "#0066CC", "W2 Severely Wet"),
    ( 1.6,     2.0, "#003366", "W3 Extremely Wet"),
    ( 2.0,   np.inf,"#001933", "W4 Exceptionally Wet"),
]
labels = [t[3] for t in thresholds]

rows = []

start_date = datetime(2020, 9, 1)   # Sept 2024
end_date = datetime(2025, 8, 1)     # Aug 2025

# Loop over months
date = start_date
while date <= end_date:
  date_str = date.strftime("%Y-%m")
  file = f"./data/spi003_{date_str}.tif"
  with rasterio.open(file) as src:
      data = src.read(1)
  valid = np.isfinite(data)

  total = valid.sum()
  out = {"month": date_str}

  for lo, hi, _, lab in thresholds:
      in_bin = (data >= lo) & (data < hi) & valid
      out[lab] = (in_bin.sum() / total) * 100.0

  rows.append(out)
  date += relativedelta(months=1)

df = pd.DataFrame(rows)
df.to_csv("../public/spi/spi3_distribution_2025.csv", index=False)
df


,month,D4 Exceptional Drought,D3 Extreme Drought,D2 Severe Drought,D1 Moderate Drought,D0 Abnormally Dry,Near Normal,W0 Abnormally Wet,W1 Moderately Wet,W2 Severely Wet,W3 Extremely Wet,W4 Exceptionally Wet
0,2020-09,0.000000,0.332659,1.643847,12.007653,16.610703,54.559471,9.201932,4.489501,1.095898,0.058337,0.000000
1,2020-10,0.689277,3.457496,5.274964,19.941802,14.520302,53.013893,1.479601,1.622665,0.000000,0.000000,0.000000
2,2020-11,0.148273,0.229180,2.294580,8.251876,9.994687,55.395631,15.670022,8.015751,0.000000,0.000000,0.000000
3,2020-12,1.089995,3.227968,3.013372,13.427876,14.534191,55.561960,5.854512,2.243882,0.769837,0.276405,0.000000
4,2021-01,0.366688,0.310435,1.436543,5.354830,11.094058,31.482067,11.184341,32.223430,5.048215,0.805603,0.693791
5,2021-02,0.000000,0.000000,1.405639,4.208582,6.510106,52.593729,25.253227,9.906140,0.122577,0.000000,0.000000
6,2021-03,0.000000,0.000000,0.000000,0.000347,1.723713,11.501373,16.375967,49.247004,10.146432,7.263970,3.741193
7,2021-04,0.000000,0.000000,0.000000,0.671567,1.813996,17.731602,28.488140,37.839734,5.763882,4.836744,2.854335
8,2021-05,0.000000,0.000000,0.000000,0.000000,0.371897,21.094301,23.491664,38.008841,8.710931,6.653518,1.668849
9,2021-06,2.423754,4.506863,5.460739,19.744915,15.393617,33.546077,11.279138,5.547550,1.541063,0.548296,0.007987
